Part 3A
Initialization & Configuration.

This part does not train LDA.

Its only responsibility is

load configuration;
load fold information;
create output folders;
create logger;
initialize experiment;
prepare for long execution;

In [83]:
# ============================================================
# TopicEvalBench
# Notebook 3
#
# Part 3A
# Initialization & Configuration
#
# This notebook trains LDA models and extracts topic
# representations for downstream evaluation.
#
# IMPORTANT
# ----------
# This notebook DOES NOT perform classification.
# Classification is performed in Notebook 4.
#
# Author : Mahedi Hasan
# ============================================================

In [84]:
!pip -q install \
gensim \
pyLDAvis \
openpyxl

In [85]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [86]:
import os
import gc
import json
import time
import pickle
import random
import logging
import warnings
import sys

from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

from tqdm.auto import tqdm

import gensim

from gensim import corpora
from gensim.models import LdaModel
from gensim.models import CoherenceModel

from datetime import datetime

print("=" * 60)
print("Python :", sys.version)
print("NumPy  :", np.__version__)
print("Pandas :", pd.__version__)
print("Sklearn:", sklearn.__version__)
print("Gensim :", gensim.__version__)
print("=" * 60)

warnings.filterwarnings("ignore")

Python : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
NumPy  : 2.0.2
Pandas : 2.2.2
Sklearn: 1.6.1
Gensim : 4.4.0


In [87]:
# ------------------------------------------------------------
# Load configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/TopicEvalBench")

CONFIG_FILE = PROJECT_ROOT/"config"/"config.json"

with open(CONFIG_FILE) as f:

    CONFIG = json.load(f)

print("Configuration Loaded")

Configuration Loaded


In [88]:
NUM_FOLDS = CONFIG["n_folds"]

MIN_TOPICS=CONFIG["topic_range"]["minimum"]

MAX_TOPICS=CONFIG["topic_range"]["maximum"]


REMOVE_BELOW = CONFIG["remove_below"]

REMOVE_ABOVE = CONFIG["remove_above"]

RANDOM_STATE=CONFIG["random_seed"]

#LDA Parameters
LDA_CHUNKSIZE=CONFIG["lda"]["chunksize"]

TOP_WORDS=CONFIG["top_words"]

LDA_PASSES=CONFIG["lda"]["passes"]

LDA_ITERATIONS=CONFIG["lda"]["iterations"]

LDA_CHUNKSIZE=CONFIG["lda"]["chunksize"]

ALPHA=CONFIG["lda"]["alpha"]

ETA=CONFIG["lda"]["eta"]

LDA_UPDATE_EVERY=CONFIG["lda"]["update_every"]

LDA_DECAY=CONFIG["lda"]["decay"]

LDA_OFFSET=CONFIG["lda"]["offset"]

MINIMUM_PROBABILITY=CONFIG["lda"]["minimum_probability"]

PER_WORD_TOPICS=CONFIG["lda"]["per_word_topics"]

EVAL_EVERY=CONFIG["lda"]["eval_every"]

DTYPE=np.float32

print("Configuration Loaded")

print(CONFIG)

Configuration Loaded
{'project_name': 'TopicEvalBench', 'dataset_name': 'TREC', 'dataset_file': 'TREC.xlsx', 'text_column': 'Text', 'label_column': 'Label', 'document_id_column': 'DocumentID', 'label_id_column': 'LabelID', 'random_seed': 42, 'n_folds': 5, 'remove_below': 5, 'remove_above': 0.5, 'topic_range': {'minimum': 2, 'maximum': 100}, 'lda': {'passes': 20, 'iterations': 400, 'chunksize': 2000, 'alpha': 'symmetric', 'eta': 'auto', 'update_every': 1, 'decay': 0.5, 'offset': 1.0, 'minimum_probability': 0.0, 'per_word_topics': False, 'eval_every': None, 'dtype': 'float32'}, 'top_words': 20, 'classifier': {'name': 'LogisticRegression', 'solver': 'lbfgs', 'max_iter': 1000, 'random_state': 42}, 'created_by': 'Notebook_01'}


In [89]:
TOPIC_LIST = list(

    range(

        MIN_TOPICS,

        MAX_TOPICS+1

    )

)

print(TOPIC_LIST[:20])

print()

print("Total Topic Numbers =",len(TOPIC_LIST))

[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]

Total Topic Numbers = 99


In [90]:
MODEL_DIR = PROJECT_ROOT/"models"

VECTOR_DIR = PROJECT_ROOT/"topic_vectors"

METRIC_DIR = PROJECT_ROOT/"intrinsic_metrics"

LOG_DIR = PROJECT_ROOT/"logs"

DICTIONARY_DIR = PROJECT_ROOT/"dictionary"

for directory in [

    MODEL_DIR,

    VECTOR_DIR,

    METRIC_DIR,

    LOG_DIR,

    DICTIONARY_DIR

]:

    directory.mkdir(

        parents=True,

        exist_ok=True

    )

print("Directories Ready")

Directories Ready


In [91]:
LOG_FILE = LOG_DIR/"Notebook3.log"

logging.basicConfig(

    filename=LOG_FILE,

    level=logging.INFO,

    format="%(asctime)s %(message)s"

)

logging.info("Notebook 3 Started")

print(LOG_FILE)

/content/drive/MyDrive/TopicEvalBench/logs/Notebook3.log


In [92]:
random.seed(RANDOM_STATE)

np.random.seed(RANDOM_STATE)

os.environ["PYTHONHASHSEED"]=str(RANDOM_STATE)

print("Seed =",RANDOM_STATE)

Seed = 42


In [93]:
def print_line():

    print("="*70)

In [94]:
def load_fold(fold):

    train_file = (

        PROJECT_ROOT/

        "folds"/

        f"Fold_{fold}"/

        "train.xlsx"

    )

    test_file = (

        PROJECT_ROOT/

        "folds"/

        f"Fold_{fold}"/

        "test.xlsx"

    )

    train = pd.read_excel(train_file)

    test = pd.read_excel(test_file)

    return train,test

In [95]:
train,test = load_fold(1)

print(train.shape)

print(test.shape)

train.head()

(240, 5)
(60, 5)


,DocumentID,Text,Label,LabelID,Tokens
0,8042410,Development of primary cultured chicken myogen...,108,0,"['development', 'primary', 'cultured', 'chicke..."
1,8132718,Constitutive centripetal transport of the acti...,108,0,"['constitutive', 'centripetal', 'transport', '..."
2,8673465,BACKGROUND: The green fluorescent protein (GFP...,108,0,"['background', 'green', 'fluorescent', 'protei..."
3,8707050,We report fluorescent resonance energy transfe...,108,0,"['report', 'fluorescent', 'resonance', 'energy..."
4,9175625,The trafficking of the androgen receptor (AR) ...,108,0,"['trafficking', 'androgen', 'receptor', 'ar', ..."


In [96]:
for fold in range(1,NUM_FOLDS+1):

    (MODEL_DIR/f"Fold_{fold}").mkdir(

        parents=True,

        exist_ok=True

    )

    (VECTOR_DIR/f"Fold_{fold}").mkdir(

        parents=True,

        exist_ok=True

    )

    (DICTIONARY_DIR/f"Fold_{fold}").mkdir(

        parents=True,

        exist_ok=True

    )

In [97]:
print_line()

print("Notebook 3")

print_line()

print("Number of folds :",NUM_FOLDS)

print("Topic Range :",MIN_TOPICS,"-",MAX_TOPICS)

print("Total Topic Numbers :",len(TOPIC_LIST))

print("LDA Passes :",LDA_PASSES)

print("Iterations :",LDA_ITERATIONS)

print_line()

Notebook 3
Number of folds : 5
Topic Range : 2 - 100
Total Topic Numbers : 99
LDA Passes : 20
Iterations : 400


In [98]:
TOTAL_JOBS = NUM_FOLDS * len(TOPIC_LIST)

print("Total LDA Models =",TOTAL_JOBS)

Total LDA Models = 495


In [99]:
gc.collect()

print("Initialization Completed Successfully.")

Initialization Completed Successfully.


Part 3B
Dictionary & Corpus Construction
Responsibilities

For each fold:

Load train/test data.
Convert token strings back to Python lists.
Build the dictionary using only the training fold.
Filter extreme words.
Save the dictionary.
Convert train/test documents into BoW using the same dictionary.
Save BoW corpora for later LDA training.

No LDA model is trained in this part.

In [100]:
# ============================================================
# Part 3B
# Dictionary & Corpus Construction
# ============================================================

print_line()
print("Part 3B : Dictionary & Corpus Construction")
print_line()

Part 3B : Dictionary & Corpus Construction


In [101]:
import ast

from gensim.corpora import Dictionary

In [102]:
def restore_tokens(token_column):
    """
    Convert token strings stored in Excel
    back to Python lists.
    """

    restored = []

    for item in token_column:

        if isinstance(item, list):

            restored.append(item)

        else:

            restored.append(ast.literal_eval(item))

    return restored

In [103]:
def build_dictionary(train_tokens):
    """
    Build dictionary using ONLY
    training documents.
    """

    dictionary = Dictionary(train_tokens)

    # Vocabulary size before filtering
    vocab_before = len(dictionary)

    dictionary.filter_extremes(
        no_below=REMOVE_BELOW,
        no_above=REMOVE_ABOVE
    )

    dictionary.compactify()

    # Vocabulary size after filtering
    vocab_after = len(dictionary)

    return dictionary, vocab_before, vocab_after

In [104]:
def create_bow(tokens, dictionary):
    """
    Convert tokenized documents
    into Bag-of-Words representation.
    """

    corpus = []

    for document in tokens:

        bow = dictionary.doc2bow(document)

        corpus.append(bow)

    return corpus

In [105]:
def save_dictionary(dictionary, fold):

    filename = (

        DICTIONARY_DIR /

        f"Fold_{fold}" /

        "dictionary.dict"

    )

    dictionary.save(str(filename))

    return filename

In [106]:
def save_corpus(corpus, filename):

    with open(filename, "wb") as f:

        pickle.dump(corpus, f)

In [107]:
import json

def save_dictionary_stats(
    dictionary,
    fold,
    train_docs,
    vocab_before,
    vocab_after,
    no_below,
    no_above
):
    """
    Save dictionary statistics for reproducibility.
    """

    stats = {

        "fold": fold,

        "num_documents_train": train_docs,

        "vocabulary_before_filtering": vocab_before,

        "vocabulary_after_filtering": vocab_after,

        "no_below": no_below,

        "no_above": no_above,

        "dictionary_size": len(dictionary),

        "random_seed": RANDOM_STATE,

        "created_by": "Notebook_03_Part3B"

    }

    output_file = (
        DICTIONARY_DIR /
        f"Fold_{fold}" /
        "dictionary_stats.json"
    )

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(
            stats,
            f,
            indent=4
        )

    return output_file

In [108]:
def process_fold(fold):

    print_line()

    print(f"Processing Fold {fold}")

    train, test = load_fold(fold)

    train_tokens = restore_tokens(train["Tokens"])

    test_tokens = restore_tokens(test["Tokens"])

    dictionary, vocab_before, vocab_after = build_dictionary(
    train_tokens)

    dictionary_file = save_dictionary(

        dictionary,

        fold

    )

    stats_file = save_dictionary_stats(

    dictionary=dictionary,

    fold=fold,

    train_docs=len(train_tokens),

    vocab_before=vocab_before,

    vocab_after=vocab_after,

    no_below=REMOVE_BELOW,

    no_above=REMOVE_ABOVE)


    train_corpus = create_bow(

        train_tokens,

        dictionary

    )

    test_corpus = create_bow(

        test_tokens,

        dictionary

    )

    corpus_dir = MODEL_DIR / f"Fold_{fold}"

    save_corpus(

        train_corpus,

        corpus_dir / "train_corpus.pkl"

    )

    save_corpus(

        test_corpus,

        corpus_dir / "test_corpus.pkl"

    )

    print(f"Vocabulary Before Filtering : {vocab_before}")

    print(f"Vocabulary After Filtering  : {vocab_after}")

    print(f"Dictionary Size             : {len(dictionary)}")

    print(f"Training Docs   : {len(train_corpus)}")

    print(f"Testing Docs    : {len(test_corpus)}")

    logging.info(

        f"Fold {fold} processed."

    )

    print()

    print("Dictionary saved to")

    print(dictionary_file)

    print()

    print("Statistics saved to")

    print(stats_file)

    return dictionary

In [109]:
for fold in range(1, NUM_FOLDS + 1):

    process_fold(fold)

Processing Fold 1
Vocabulary Before Filtering : 4051
Vocabulary After Filtering  : 891
Dictionary Size             : 891
Training Docs   : 240
Testing Docs    : 60

Dictionary saved to
/content/drive/MyDrive/TopicEvalBench/dictionary/Fold_1/dictionary.dict

Statistics saved to
/content/drive/MyDrive/TopicEvalBench/dictionary/Fold_1/dictionary_stats.json
Processing Fold 2
Vocabulary Before Filtering : 4106
Vocabulary After Filtering  : 860
Dictionary Size             : 860
Training Docs   : 240
Testing Docs    : 60

Dictionary saved to
/content/drive/MyDrive/TopicEvalBench/dictionary/Fold_2/dictionary.dict

Statistics saved to
/content/drive/MyDrive/TopicEvalBench/dictionary/Fold_2/dictionary_stats.json
Processing Fold 3
Vocabulary Before Filtering : 4136
Vocabulary After Filtering  : 857
Dictionary Size             : 857
Training Docs   : 240
Testing Docs    : 60

Dictionary saved to
/content/drive/MyDrive/TopicEvalBench/dictionary/Fold_3/dictionary.dict

Statistics saved to
/content/d

In [110]:
dictionary = Dictionary.load(

    str(

        DICTIONARY_DIR /

        "Fold_1" /

        "dictionary.dict"

    )

)

print("Dictionary Size :", len(dictionary))

print()

for token_id in range(min(20, len(dictionary))):

    print(token_id, "->", dictionary[token_id])

Dictionary Size : 891

0 -> absence
1 -> accumulation
2 -> addition
3 -> along
4 -> among
5 -> analysis
6 -> cellular
7 -> chain
8 -> change
9 -> chicken
10 -> cultured
11 -> cycle
12 -> development
13 -> differentiation
14 -> distribution
15 -> early
16 -> expressed
17 -> expression
18 -> first
19 -> found


In [111]:
with open(

    MODEL_DIR /

    "Fold_1" /

    "train_corpus.pkl",

    "rb"

) as f:

    corpus = pickle.load(f)

print("Number of Documents :", len(corpus))

print()

print("First Document")

print(corpus[0])

Number of Documents : 240

First Document
[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1), (12, 1), (13, 2), (14, 1), (15, 1), (16, 3), (17, 1), (18, 3), (19, 2), (20, 1), (21, 1), (22, 1), (23, 1), (24, 1), (25, 2), (26, 2), (27, 1), (28, 1), (29, 1), (30, 2), (31, 1), (32, 1), (33, 1), (34, 1), (35, 1), (36, 2), (37, 1), (38, 1), (39, 1), (40, 1), (41, 1), (42, 1), (43, 1), (44, 1), (45, 1), (46, 1), (47, 1), (48, 2), (49, 1), (50, 1), (51, 1), (52, 1), (53, 1)]


In [112]:
print_line()

print("Dictionary & Corpus Construction Completed")

print()

print(f"Processed Folds : {NUM_FOLDS}")

print(f"Dictionary saved in : {DICTIONARY_DIR}")

print(f"Corpora saved in : {MODEL_DIR}")

print_line()

Dictionary & Corpus Construction Completed

Processed Folds : 5
Dictionary saved in : /content/drive/MyDrive/TopicEvalBench/dictionary
Corpora saved in : /content/drive/MyDrive/TopicEvalBench/models


In [113]:
dictionary_sizes = []

for fold in range(1, NUM_FOLDS + 1):

    dictionary = Dictionary.load(
        str(
            DICTIONARY_DIR /
            f"Fold_{fold}" /
            "dictionary.dict"
        )
    )

    dictionary_sizes.append(len(dictionary))

summary = pd.DataFrame({
    "Fold": range(1, NUM_FOLDS + 1),
    "Dictionary_Size": dictionary_sizes
})

print(summary)

print("\nMean Dictionary Size :", summary["Dictionary_Size"].mean())
print("Std Dev :", summary["Dictionary_Size"].std())
print("Min :", summary["Dictionary_Size"].min())
print("Max :", summary["Dictionary_Size"].max())

   Fold  Dictionary_Size
0     1              891
1     2              860
2     3              857
3     4              886
4     5              882

Mean Dictionary Size : 875.2
Std Dev : 15.61089363233252
Min : 857
Max : 891


Notebook 3C-1,

This section only prepares everything for LDA training. No LDA model is trained yet.

In [114]:
# ============================================================
# Part 3C
# LDA Model Training
#
# Responsibilities
# ----------------
# 1. Load dictionary
# 2. Load corpus
# 3. Train LDA
# 4. Save model
# 5. Save metadata
# 6. Save Phi matrix
# 7. Save Theta matrix
# 8. Save topic statistics
#
# No intrinsic evaluation here.
# No classification here.
# ============================================================

print_line()
print("Part 3C : LDA Model Training")
print_line()

Part 3C : LDA Model Training


In [115]:
import os
import gc
import json
import time
import pickle

import numpy as np
import pandas as pd

from pathlib import Path

from tqdm.auto import tqdm

from gensim.models import LdaModel
from gensim.corpora import Dictionary

In [116]:
def load_dictionary(fold):
    """
    Load dictionary of a fold.
    """

    dictionary_file = (
        DICTIONARY_DIR /
        f"Fold_{fold}" /
        "dictionary.dict"
    )

    dictionary = Dictionary.load(str(dictionary_file))

    return dictionary

In [117]:
def load_corpus(fold):
    """
    Load train and test corpus.
    """

    corpus_dir = MODEL_DIR / f"Fold_{fold}"

    with open(corpus_dir / "train_corpus.pkl", "rb") as f:
        train_corpus = pickle.load(f)

    with open(corpus_dir / "test_corpus.pkl", "rb") as f:
        test_corpus = pickle.load(f)

    return train_corpus, test_corpus

In [118]:
def load_training_dataframe(fold):

    train_df, _ = load_fold(fold)

    return train_df

In [119]:
def load_testing_dataframe(fold):

    _, test_df = load_fold(fold)

    return test_df

In [120]:
def create_model_directory(fold, K):

    model_dir = (
        MODEL_DIR /
        f"Fold_{fold}" /
        f"K_{K:03d}"
    )

    model_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    return model_dir

In [121]:
training_summary = []

In [122]:
fold = 1

dictionary = load_dictionary(fold)

train_corpus, test_corpus = load_corpus(fold)

train_df = load_training_dataframe(fold)

print("Dictionary Size :", len(dictionary))

print("Training Docs   :", len(train_corpus))

print("Testing Docs    :", len(test_corpus))

print("Train DataFrame :", train_df.shape)

Dictionary Size : 891
Training Docs   : 240
Testing Docs    : 60
Train DataFrame : (240, 5)


In [123]:
folder = create_model_directory(1,20)

print(folder)

/content/drive/MyDrive/TopicEvalBench/models/Fold_1/K_020


In [124]:
gc.collect()

print("Part 3C-1 Completed Successfully.")

Part 3C-1 Completed Successfully.


Notebook 3C-2

Train and Save One LDA Model

In [125]:
import gensim

def save_metadata(
    model_dir,
    fold,
    K,
    dictionary,
    train_corpus,
    lda,
    training_time
):
    """
    Save metadata for one trained LDA model.
    """

    metadata = {

    # -----------------------------
    # Dataset
    # -----------------------------
    "fold": fold,
    "topics": K,
    "documents": len(train_corpus),
    "dictionary_size": len(dictionary),

    # -----------------------------
    # Training Parameters
    # -----------------------------
    "passes": LDA_PASSES,
    "iterations": LDA_ITERATIONS,
    "chunksize": LDA_CHUNKSIZE,

    "alpha": (
        lda.alpha.tolist()
        if hasattr(lda.alpha, "tolist")
        else str(lda.alpha)
    ),

    "eta": str(lda.eta),

    "update_every": LDA_UPDATE_EVERY,
    "decay": LDA_DECAY,
    "offset": LDA_OFFSET,

    "minimum_probability": MINIMUM_PROBABILITY,

    "per_word_topics": PER_WORD_TOPICS,

    "eval_every": EVAL_EVERY,

    "dtype": str(DTYPE),

    # -----------------------------
    # Reproducibility
    # -----------------------------
    "random_seed": RANDOM_STATE,
    "gensim_version": gensim.__version__,

    # -----------------------------
    # Performance
    # -----------------------------
    "training_time_seconds": round(training_time, 4),

    "log_perplexity_train":
        float(
            lda.log_perplexity(train_corpus)
        ),

    # -----------------------------
    # Notebook
    # -----------------------------
    "created_by": "Notebook_03_Part3C",

    "model_path": str(model_dir / "lda.model"),
    "theta_train_file": str(model_dir / "theta_train.csv"),
    "theta_test_file": str(model_dir / "theta_test.csv"),
    "phi_path": str(model_dir / "phi.csv"),
    "top_words_path": str(model_dir / "top_words.csv"),
    "topic_sizes_path": str(model_dir / "topic_sizes.csv"),

    "dictionary_path": str(
    DICTIONARY_DIR /
    f"Fold_{fold}" /
    "dictionary.dict"),

    "train_corpus_path": str(
    MODEL_DIR /
    f"Fold_{fold}" /
    "train_corpus.pkl"),
    }

    output_file = model_dir / "metadata.json"

    with open(output_file, "w", encoding="utf-8") as f:

        json.dump(
            metadata,
            f,
            indent=4
        )

In [126]:
def train_single_lda(
    fold,
    K
):
    """
    Train one LDA model.

    Returns
    -------
    lda
        Trained gensim LDA model.
    """

    dictionary = load_dictionary(fold)

    train_corpus, _ = load_corpus(fold)

    model_dir = create_model_directory(
        fold,
        K
    )

    print(f"Training Fold {fold}  K={K}")

    start_time = time.time()


    lda = LdaModel(

    corpus=train_corpus,

    id2word=dictionary,

    num_topics=K,

    random_state=RANDOM_STATE,

    passes=LDA_PASSES,

    iterations=LDA_ITERATIONS,

    chunksize=LDA_CHUNKSIZE,

    alpha=ALPHA,

    eta=ETA,

    update_every=LDA_UPDATE_EVERY,

    decay=LDA_DECAY,

    offset=LDA_OFFSET,

    eval_every=EVAL_EVERY,

    minimum_probability=MINIMUM_PROBABILITY,

    per_word_topics=PER_WORD_TOPICS,

    dtype=DTYPE
    )

    training_time = time.time() - start_time

    lda.save(
        str(
            model_dir /
            "lda.model"
        )
    )

    save_metadata(

        model_dir=model_dir,

        fold=fold,

        K=K,

        dictionary=dictionary,

        train_corpus=train_corpus,

        lda=lda,

        training_time=training_time

    )

    print(f"Finished in {training_time:.2f} seconds")

    return lda

In [127]:
lda = train_single_lda(

    fold=1,

    K=5

)

Training Fold 1  K=5
Finished in 6.02 seconds


In [128]:
model = LdaModel.load(

    str(

        MODEL_DIR /

        "Fold_1" /

        "K_005" /

        "lda.model"

    )

)

print(model)

LdaModel<num_terms=891, num_topics=5, decay=0.5, chunksize=2000>


In [129]:
metadata_file = (

    MODEL_DIR /

    "Fold_1" /

    "K_005" /

    "metadata.json"

)

with open(
    metadata_file,
    "r"
) as f:

    metadata = json.load(f)

metadata

{'fold': 1,
 'topics': 5,
 'documents': 240,
 'dictionary_size': 891,
 'passes': 20,
 'iterations': 400,
 'chunksize': 2000,
 'alpha': [0.20000000298023224,
  0.20000000298023224,
  0.20000000298023224,
  0.20000000298023224,
  0.20000000298023224],
 'eta': '[0.34800598 0.5017559  0.5347968  0.21995631 0.45029557 1.7414714\n 0.33190185 0.38513726 0.607558   0.1790911  0.24233264 0.38946405\n 0.65419203 0.30550027 0.30878255 0.75669134 0.4086989  1.7628483\n 0.34840432 1.6783191  0.18046997 0.3386618  0.21368214 0.2848632\n 0.29833418 0.3534899  0.21227327 1.1514271  0.43806693 0.26938173\n 1.5476742  0.22286184 0.49046603 0.6305704  1.1847092  0.43884432\n 0.6170383  0.22473352 0.3038548  0.20269312 0.67921674 0.3426452\n 0.47925767 0.26342994 0.20819435 0.47572625 0.24663693 0.8121762\n 0.3148815  0.21663071 0.32881653 0.20556524 0.9010773  1.3890481\n 0.32977772 0.35762116 1.0092175  0.20782049 1.8375325  0.6576163\n 0.4325214  0.23492591 0.25143126 0.2698252  0.27701473 0.29718402\n

In [130]:
del lda
del model

gc.collect()

print("Part 3C-2 Completed Successfully.")

Part 3C-2 Completed Successfully.


Part 3C-3

Responsibilities of 3C-3

For each trained LDA model, we will generate and save:

θ (Theta): Document–Topic Distribution (theta_train.csv)
Φ (Phi): Topic–Word Distribution (phi.csv)
Topic Size Statistics (topic_sizes.csv)

No new model is trained in this part.

In [131]:
# ============================================================
# Part 3C-3
# Extract LDA Representations
#
# 1. Theta (Document-Topic Matrix)
# 2. Phi (Topic-Word Matrix)
# 3. Topic Size Statistics
# ============================================================

print_line()
print("Part 3C-3 : Extract LDA Representations")
print_line()

Part 3C-3 : Extract LDA Representations


In [132]:
def load_lda_model(fold, K):
    """
    Load a trained LDA model.
    """

    model_path = (
        MODEL_DIR /
        f"Fold_{fold}" /
        f"K_{K:03d}" /
        "lda.model"
    )

    lda = LdaModel.load(str(model_path))

    return lda

In [133]:
def save_theta_train(
    lda,
    train_corpus,
    train_df,
    model_dir):
    """
    Save document-topic probability matrix.
    """

    theta = []

    for bow in train_corpus:

        topic_vector = lda.get_document_topics(
            bow,
            minimum_probability=0.0
        )

        topic_vector = [
            probability
            for _, probability in topic_vector
        ]

        theta.append(topic_vector)

    theta = np.array(theta)

    columns = [
        f"Topic_{i+1}"
        for i in range(theta.shape[1])
    ]

    theta_df = pd.DataFrame(
        theta,
        columns=columns
    )

    # Keep document information
    theta_df.insert(
        0,
        "LabelID",
        train_df["LabelID"].values
    )

    theta_df.insert(
        0,
        "Label",
        train_df["Label"].values
    )

    theta_df.insert(
        0,
        "DocumentID",
        train_df["DocumentID"].values
    )

    output_file = model_dir / "theta_train.csv"

    theta_df.to_csv(
        output_file,
        index=False
    )

    return theta_df

In [134]:
def save_theta_test(
    lda,
    test_corpus,
    test_df,
    model_dir):
    """
    Save document-topic probability matrix for the testing corpus.
    """

    theta = []

    for bow in test_corpus:

        topic_vector = lda.get_document_topics(
            bow,
            minimum_probability=0.0
        )

        topic_vector = [
            probability
            for _, probability in topic_vector
        ]

        theta.append(topic_vector)

    theta = np.array(theta)

    columns = [
        f"Topic_{i+1}"
        for i in range(theta.shape[1])
    ]

    theta_df = pd.DataFrame(
        theta,
        columns=columns
    )

    # Keep document information
    theta_df.insert(
        0,
        "LabelID",
        test_df["LabelID"].values
    )

    theta_df.insert(
        0,
        "Label",
        test_df["Label"].values
    )

    theta_df.insert(
        0,
        "DocumentID",
        test_df["DocumentID"].values
    )

    output_file = model_dir / "theta_test.csv"

    theta_df.to_csv(
        output_file,
        index=False
    )

    return theta_df

In [135]:
def save_phi(
    lda,
    dictionary,
    model_dir):
    """
    Save topic-word probability matrix.
    """

    phi = lda.get_topics()

    vocabulary = [
        dictionary[i]
        for i in range(len(dictionary))
    ]

    phi_df = pd.DataFrame(
        phi,
        columns=vocabulary
    )

    phi_df.index = [
        f"Topic_{i+1}"
        for i in range(phi.shape[0])
    ]

    output_file = model_dir / "phi.csv"

    phi_df.to_csv(output_file)

    return phi_df

In [136]:
def save_top_words(
    lda,
    model_dir,
    topn=20):
    """
    Save the top-N words for each topic.

    Parameters
    ----------
    lda : gensim.models.LdaModel
        Trained LDA model.

    model_dir : pathlib.Path
        Output directory.

    topn : int
        Number of top words per topic.
    """

    rows = []

    for topic_id in range(lda.num_topics):

        words = lda.show_topic(
            topic_id,
            topn=topn
        )

        for rank, (word, prob) in enumerate(words, start=1):

            # Get the dictionary ID of the word
            word_id = lda.id2word.token2id[word]

            rows.append({

                "TopicID": topic_id,

                "Topic": f"Topic_{topic_id+1}",

                "Rank": rank,

                "WordID": word_id,

                "Word": word,

                "Probability": prob

            })

    top_words_df = pd.DataFrame(rows)

    output_file = model_dir / "top_words.csv"

    top_words_df.to_csv(
        output_file,
        index=False
    )

    return top_words_df

In [137]:
def save_topic_sizes(
    theta_df,
    model_dir):
    """
    Compute dominant topic counts.
    """

    topic_columns = [
        c for c in theta_df.columns
        if c.startswith("Topic_")
    ]

    dominant_topics = (
        theta_df[topic_columns]
        .idxmax(axis=1)
    )

    topic_sizes = (
        dominant_topics
        .value_counts()
        .sort_index()
    )

    topic_size_df = pd.DataFrame({

        "Topic": topic_sizes.index,

        "Documents": topic_sizes.values

    })

    output_file = (
        model_dir /
        "topic_sizes.csv"
    )

    topic_size_df.to_csv(
        output_file,
        index=False
    )

    return topic_size_df

In [138]:
def extract_lda_outputs(
    fold,
    K):
    """
    Extract theta, phi and topic sizes.
    """

    model_dir = create_model_directory(
        fold,
        K
    )

    dictionary = load_dictionary(fold)

    train_corpus, test_corpus = load_corpus(fold)

    train_df = load_training_dataframe(fold)
    test_df = load_testing_dataframe(fold)

    lda = load_lda_model(
        fold,
        K
    )

    theta_df = save_theta_train(
        lda,
        train_corpus,
        train_df,
        model_dir
    )

    # Save document-topic distributions for testing documents
    theta_test_df = save_theta_test(
        lda=lda,
        test_corpus=test_corpus,
        test_df=test_df,
        model_dir=model_dir
)

    print("="*60)
    print("Fold:", fold)
    print("K:", K)

    print("Dictionary Size :", len(dictionary))
    print("Max Dictionary ID:", max(dictionary.keys()))

    print("LDA Topic Matrix :", lda.get_topics().shape)

    print("="*60)

    phi_df = save_phi(
        lda,
        dictionary,
        model_dir
    )

    top_words_df = save_top_words(
        lda,
        model_dir,
        topn=20
    )

    topic_size_df = save_topic_sizes(
        theta_df,
        model_dir
    )

    print("Theta Shape      :", theta_df.shape)

    print("Phi Shape        :", phi_df.shape)

    print("Top Words Rows   :", len(top_words_df))

    print("Topic Statistics :")

    print(topic_size_df.head())

    del lda

    gc.collect()

In [139]:
extract_lda_outputs(
    fold=1,
    K=5)

Fold: 1
K: 5
Dictionary Size : 891
Max Dictionary ID: 890
LDA Topic Matrix : (5, 891)
Theta Shape      : (240, 8)
Phi Shape        : (5, 891)
Top Words Rows   : 100
Topic Statistics :
     Topic  Documents
0  Topic_1         50
1  Topic_2         34
2  Topic_3         71
3  Topic_4         54
4  Topic_5         31


Part 3C-4
Purpose

For each fold:

Train all LDA models;
Save metadata;
Save theta;
Save phi;
Save top words;
Save topic sizes;
Record training summary;
Continue if one topic fails

In [140]:
def process_fold(fold):
    """
    Train all LDA models for one fold.

    Parameters
    ----------
    fold : int
        Fold number (1-5)

    Returns
    -------
    summary_df : pandas.DataFrame
    """

    print_line()
    print(f"Processing Fold {fold}")
    print_line()

    fold_summary = []

    for K in tqdm(
        range(MIN_TOPICS, MAX_TOPICS + 1),
        desc=f"Fold {fold}"
    ):

        model_dir = (
            MODEL_DIR /
            f"Fold_{fold}" /
            f"K_{K:03d}"
        )

        # --------------------------------------------------
        # Required output files
        # --------------------------------------------------

        required_files = [

            model_dir / "lda.model",

            model_dir / "metadata.json",

            model_dir / "theta_train.csv",

            model_dir / "theta_test.csv",

            model_dir / "phi.csv",

            model_dir / "topic_sizes.csv",

            model_dir / "top_words.csv"

        ]

        # --------------------------------------------------
        # Resume capability
        # --------------------------------------------------

        if all(f.exists() for f in required_files):

            print(f"Fold {fold} | K={K:03d} already completed. Skipping.")

            try:

                with open(
                    model_dir / "metadata.json",
                    "r",
                    encoding="utf-8"
                ) as f:

                    metadata = json.load(f)

                fold_summary.append({

                    "Fold": fold,

                    "Topics": K,

                    "DictionarySize":
                        metadata["dictionary_size"],

                    "Documents":
                        metadata["documents"],

                    "TrainingTime":
                        metadata["training_time_seconds"],

                    "LogPerplexityTrain":
                        metadata["log_perplexity_train"],

                    "Status": "Already Exists"

                })

            except Exception:

                fold_summary.append({

                    "Fold": fold,

                    "Topics": K,

                    "DictionarySize": np.nan,

                    "Documents": np.nan,

                    "TrainingTime": np.nan,

                    "LogPerplexityTrain": np.nan,

                    "Status": "Already Exists"

                })

            continue

        # --------------------------------------------------
        # Train Model
        # --------------------------------------------------

        try:

            lda = train_single_lda(
                fold,
                K
            )

            extract_lda_outputs(
                fold,
                K
            )

            with open(
                model_dir / "metadata.json",
                "r",
                encoding="utf-8"
            ) as f:

                metadata = json.load(f)

            fold_summary.append({

                "Fold": fold,

                "Topics": K,

                "DictionarySize":
                    metadata["dictionary_size"],

                "Documents":
                    metadata["documents"],

                "TrainingTime":
                    metadata["training_time_seconds"],

                "LogPerplexityTrain":
                    metadata["log_perplexity_train"],

                "Status": "Success"

            })

            del lda

            gc.collect()

        except Exception as e:

            print(f"\nError while processing Fold={fold}, K={K}")

            print(e)

            fold_summary.append({

                "Fold": fold,

                "Topics": K,

                "DictionarySize": np.nan,

                "Documents": np.nan,

                "TrainingTime": np.nan,

                "LogPerplexityTrain": np.nan,

                "Status": str(e)

            })

            gc.collect()

            continue

    summary_df = pd.DataFrame(fold_summary)

    summary_file = (
        MODEL_DIR /
        f"Fold_{fold}" /
        "training_summary.csv"
    )

    summary_df.to_csv(
        summary_file,
        index=False
    )

    print("\nFold Summary")
    print(summary_df.head())

    return summary_df

In [141]:
summary_fold1 = process_fold(
    fold=1
)

Processing Fold 1


Fold 1:   0%|          | 0/99 [00:00<?, ?it/s]

Training Fold 1  K=2
Finished in 3.31 seconds
Fold: 1
K: 2
Dictionary Size : 891
Max Dictionary ID: 890
LDA Topic Matrix : (2, 891)
Theta Shape      : (240, 5)
Phi Shape        : (2, 891)
Top Words Rows   : 40
Topic Statistics :
     Topic  Documents
0  Topic_1        124
1  Topic_2        116
Training Fold 1  K=3
Finished in 4.63 seconds
Fold: 1
K: 3
Dictionary Size : 891
Max Dictionary ID: 890
LDA Topic Matrix : (3, 891)
Theta Shape      : (240, 6)
Phi Shape        : (3, 891)
Top Words Rows   : 60
Topic Statistics :
     Topic  Documents
0  Topic_1         97
1  Topic_2         70
2  Topic_3         73
Training Fold 1  K=4
Finished in 2.79 seconds
Fold: 1
K: 4
Dictionary Size : 891
Max Dictionary ID: 890
LDA Topic Matrix : (4, 891)
Theta Shape      : (240, 7)
Phi Shape        : (4, 891)
Top Words Rows   : 80
Topic Statistics :
     Topic  Documents
0  Topic_1         67
1  Topic_2         44
2  Topic_3         61
3  Topic_4         68
Fold 1 | K=005 already completed. Skipping.
Train

In [142]:
summary_fold1.head()

,Fold,Topics,DictionarySize,Documents,TrainingTime,LogPerplexityTrain,Status
0,1,2,891,240,3.3123,-6.298535,Success
1,1,3,891,240,4.6282,-6.214306,Success
2,1,4,891,240,2.7942,-6.193754,Success
3,1,5,891,240,6.0153,-6.191917,Already Exists
4,1,6,891,240,2.5762,-6.224693,Success


In [143]:
print_line()

print("Models Trained :",
      len(summary_fold1))

print()

print(summary_fold1["TrainingTime"].describe())

print()

print("Failures :")

print(
    (summary_fold1["Status"] != "Success").sum()
)

print_line()

Models Trained : 99

count    99.000000
mean      5.285724
std       3.243076
min       2.415700
25%       3.052850
50%       3.947000
75%       6.089700
max      12.816600
Name: TrainingTime, dtype: float64

Failures :
1


In [144]:
gc.collect()

print("Part 3C-4 Completed Successfully.")

Part 3C-4 Completed Successfully.


Part 3C-5

Responsibilities,
Run all 5 folds,
Merge all fold summaries,
Validate outputs,
Save training_summary_all_folds.csv,
Produce an experiment report

In [145]:
print_line()
print("Starting LDA Training for All Folds")
print_line()

all_fold_summaries = []

for fold in range(1, NUM_FOLDS + 1):

    summary_df = process_fold(fold)

    all_fold_summaries.append(summary_df)

print()

print("All folds completed.")

Starting LDA Training for All Folds
Processing Fold 1


Fold 1:   0%|          | 0/99 [00:00<?, ?it/s]

Fold 1 | K=002 already completed. Skipping.
Fold 1 | K=003 already completed. Skipping.
Fold 1 | K=004 already completed. Skipping.
Fold 1 | K=005 already completed. Skipping.
Fold 1 | K=006 already completed. Skipping.
Fold 1 | K=007 already completed. Skipping.
Fold 1 | K=008 already completed. Skipping.
Fold 1 | K=009 already completed. Skipping.
Fold 1 | K=010 already completed. Skipping.
Fold 1 | K=011 already completed. Skipping.
Fold 1 | K=012 already completed. Skipping.
Fold 1 | K=013 already completed. Skipping.
Fold 1 | K=014 already completed. Skipping.
Fold 1 | K=015 already completed. Skipping.
Fold 1 | K=016 already completed. Skipping.
Fold 1 | K=017 already completed. Skipping.
Fold 1 | K=018 already completed. Skipping.
Fold 1 | K=019 already completed. Skipping.
Fold 1 | K=020 already completed. Skipping.
Fold 1 | K=021 already completed. Skipping.
Fold 1 | K=022 already completed. Skipping.
Fold 1 | K=023 already completed. Skipping.
Fold 1 | K=024 already completed

Fold 2:   0%|          | 0/99 [00:00<?, ?it/s]

Training Fold 2  K=2
Finished in 4.80 seconds
Fold: 2
K: 2
Dictionary Size : 860
Max Dictionary ID: 859
LDA Topic Matrix : (2, 860)
Theta Shape      : (240, 5)
Phi Shape        : (2, 860)
Top Words Rows   : 40
Topic Statistics :
     Topic  Documents
0  Topic_1        120
1  Topic_2        120
Training Fold 2  K=3
Finished in 2.59 seconds
Fold: 2
K: 3
Dictionary Size : 860
Max Dictionary ID: 859
LDA Topic Matrix : (3, 860)
Theta Shape      : (240, 6)
Phi Shape        : (3, 860)
Top Words Rows   : 60
Topic Statistics :
     Topic  Documents
0  Topic_1         77
1  Topic_2         95
2  Topic_3         68
Training Fold 2  K=4
Finished in 2.34 seconds
Fold: 2
K: 4
Dictionary Size : 860
Max Dictionary ID: 859
LDA Topic Matrix : (4, 860)
Theta Shape      : (240, 7)
Phi Shape        : (4, 860)
Top Words Rows   : 80
Topic Statistics :
     Topic  Documents
0  Topic_1         50
1  Topic_2         78
2  Topic_3         67
3  Topic_4         45
Training Fold 2  K=5
Finished in 2.50 seconds
Fol

...

Training Fold 3  K=2
Finished in 3.68 seconds
Fold: 3
K: 2
Dictionary Size : 857
Max Dictionary ID: 856
LDA Topic Matrix : (2, 857)
Theta Shape      : (240, 5)
Phi Shape        : (2, 857)
Top Words Rows   : 40
Topic Statistics :
     Topic  Documents
0  Topic_1        106
1  Topic_2        134
Training Fold 3  K=3
Finished in 3.79 seconds
Fold: 3
K: 3
Dictionary Size : 857
Max Dictionary ID: 856
LDA Topic Matrix : (3, 857)
Theta Shape      : (240, 6)
Phi Shape        : (3, 857)
Top Words Rows   : 60
Topic Statistics :
     Topic  Documents
0  Topic_1        100
1  Topic_2         68
2  Topic_3         72
Training Fold 3  K=4
Finished in 2.54 seconds
Fold: 3
K: 4
Dictionary Size : 857
Max Dictionary ID: 856
LDA Topic Matrix : (4, 857)
Theta Shape      : (240, 7)
Phi Shape        : (4, 857)
Top Words Rows   : 80
Topic Statistics :
     Topic  Documents
0  Topic_1         93
1  Topic_2         53
2  Topic_3         63
3  Topic_4         31
Training Fold 3  K=5
Finished in 2.58 seconds
Fol

Fold 4:   0%|          | 0/99 [00:00<?, ?it/s]

Training Fold 4  K=2
Finished in 3.47 seconds
Fold: 4
K: 2
Dictionary Size : 886
Max Dictionary ID: 885
LDA Topic Matrix : (2, 886)
Theta Shape      : (240, 5)
Phi Shape        : (2, 886)
Top Words Rows   : 40
Topic Statistics :
     Topic  Documents
0  Topic_1        133
1  Topic_2        107
Training Fold 4  K=3
Finished in 4.60 seconds
Fold: 4
K: 3
Dictionary Size : 886
Max Dictionary ID: 885
LDA Topic Matrix : (3, 886)
Theta Shape      : (240, 6)
Phi Shape        : (3, 886)
Top Words Rows   : 60
Topic Statistics :
     Topic  Documents
0  Topic_1         95
1  Topic_2         78
2  Topic_3         67
Training Fold 4  K=4
Finished in 2.42 seconds
Fold: 4
K: 4
Dictionary Size : 886
Max Dictionary ID: 885
LDA Topic Matrix : (4, 886)
Theta Shape      : (240, 7)
Phi Shape        : (4, 886)
Top Words Rows   : 80
Topic Statistics :
     Topic  Documents
0  Topic_1         72
1  Topic_2         30
2  Topic_3         44
3  Topic_4         94
Training Fold 4  K=5
Finished in 2.38 seconds
Fol

Fold 5:   0%|          | 0/99 [00:00<?, ?it/s]

Training Fold 5  K=2
Finished in 3.01 seconds
Fold: 5
K: 2
Dictionary Size : 882
Max Dictionary ID: 881
LDA Topic Matrix : (2, 882)
Theta Shape      : (240, 5)
Phi Shape        : (2, 882)
Top Words Rows   : 40
Topic Statistics :
     Topic  Documents
0  Topic_1        141
1  Topic_2         99
Training Fold 5  K=3
Finished in 2.76 seconds
Fold: 5
K: 3
Dictionary Size : 882
Max Dictionary ID: 881
LDA Topic Matrix : (3, 882)
Theta Shape      : (240, 6)
Phi Shape        : (3, 882)
Top Words Rows   : 60
Topic Statistics :
     Topic  Documents
0  Topic_1         53
1  Topic_2         73
2  Topic_3        114
Training Fold 5  K=4
Finished in 3.33 seconds
Fold: 5
K: 4
Dictionary Size : 882
Max Dictionary ID: 881
LDA Topic Matrix : (4, 882)
Theta Shape      : (240, 7)
Phi Shape        : (4, 882)
Top Words Rows   : 80
Topic Statistics :
     Topic  Documents
0  Topic_1         28
1  Topic_2         71
2  Topic_3         72
3  Topic_4         69
Training Fold 5  K=5
Finished in 3.33 seconds
Fol

In [146]:
overall_summary = pd.concat(
    all_fold_summaries,
    ignore_index=True
)

overall_summary_file = (
    MODEL_DIR /
    "training_summary_all_folds.csv"
)

overall_summary.to_csv(
    overall_summary_file,
    index=False
)

print("Overall summary saved.")
print(overall_summary.shape)

Overall summary saved.
(495, 7)


In [147]:
# ============================================================
# Overall Training Statistics
# ============================================================

print_line()
print("Overall Training Statistics")
print_line()

# ------------------------------------------------------------
# Calculate statistics
# ------------------------------------------------------------

total_models = len(overall_summary)

new_models = (
    overall_summary["Status"] == "Success"
).sum()

existing_models = (
    overall_summary["Status"] == "Already Exists"
).sum()

successful_models = new_models + existing_models

failed_models = (
    ~overall_summary["Status"].isin(
        ["Success", "Already Exists"]
    )
).sum()

average_training_time = (
    overall_summary["TrainingTime"]
    .dropna()
    .mean()
)

# ------------------------------------------------------------
# Print statistics
# ------------------------------------------------------------

print(f"Total Models             : {total_models}")

print(f"Newly Trained            : {new_models}")

print(f"Already Existing         : {existing_models}")

print(f"Successful Models        : {successful_models}")

print(f"Failed Models            : {failed_models}")

print()

print(
    f"Success Rate             : "
    f"{100 * successful_models / total_models:.2f}%"
)

print(
    f"Failure Rate             : "
    f"{100 * failed_models / total_models:.2f}%"
)

print()

if pd.notna(average_training_time):

    print(
        f"Average Training Time    : "
        f"{average_training_time:.2f} seconds/model"
    )

else:

    print(
        "Average Training Time    : N/A"
    )

print_line()

Overall Training Statistics
Total Models             : 495
Newly Trained            : 396
Already Existing         : 99
Successful Models        : 495
Failed Models            : 0

Success Rate             : 100.00%
Failure Rate             : 0.00%

Average Training Time    : 5.19 seconds/model


In [148]:
missing = []

for fold in range(1, NUM_FOLDS + 1):

    for K in range(
        MIN_TOPICS,
        MAX_TOPICS + 1
    ):

        model_dir = (
            MODEL_DIR /
            f"Fold_{fold}" /
            f"K_{K:03d}"
        )

        required = [

            "lda.model",

            "metadata.json",

            "theta_train.csv",

            "phi.csv",

            "topic_sizes.csv",

            "top_words.csv"

        ]

        for file in required:

            if not (model_dir / file).exists():

                missing.append(

                    [fold,K,file]

                )

missing_df = pd.DataFrame(

    missing,

    columns=[

        "Fold",

        "Topics",

        "Missing File"

    ]

)

print()

print("Missing Files")

print(len(missing_df))

missing_df.head()


Missing Files
0


,Fold,Topics,Missing File


In [149]:
# ============================================================
# Save Validation Report and Experiment Summary
# ============================================================

# ------------------------------------------------------------
# Save validation report (CSV)
# ------------------------------------------------------------

validation_file = (
    MODEL_DIR /
    "validation_report.csv"
)

missing_df.to_csv(
    validation_file,
    index=False
)

print(f"Validation report saved to:\n{validation_file}")

# ------------------------------------------------------------
# Calculate experiment statistics
# ------------------------------------------------------------

total_models = len(overall_summary)

successful_models = (
    overall_summary["Status"]
    .isin(["Success", "Already Exists"])
    .sum()
)

failed_models = (
    ~overall_summary["Status"]
    .isin(["Success", "Already Exists"])
).sum()

average_training_time = float(
    overall_summary["TrainingTime"].mean()
)

# ------------------------------------------------------------
# Save experiment summary (JSON)
# ------------------------------------------------------------

experiment_summary = {

    # Dataset
    "n_folds": NUM_FOLDS,
    "min_topics": MIN_TOPICS,
    "max_topics": MAX_TOPICS,

    # Experiment statistics
    "total_models": total_models,
    "successful_models": int(successful_models),
    "failed_models": int(failed_models),

    # Performance
    "average_training_time_seconds":
        round(average_training_time, 4),

    # Validation
    "missing_output_files":
        int(len(missing_df)),

    #Date and Time of Experiment
    "completed_at":datetime.now().strftime("%Y-%m-%d %H:%M:%S"),

    # Notebook
    "created_by": "Notebook_03_Part3C"

}

experiment_summary_file = (
    MODEL_DIR /
    "experiment_summary.json"
)

with open(
    experiment_summary_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        experiment_summary,
        f,
        indent=4
    )

print(f"Experiment summary saved to:\n{experiment_summary_file}")

Validation report saved to:
/content/drive/MyDrive/TopicEvalBench/models/validation_report.csv
Experiment summary saved to:
/content/drive/MyDrive/TopicEvalBench/models/experiment_summary.json


In [150]:
# ============================================================
# Create Master Experiment Manifest
# ============================================================

manifest_rows = []

for fold in range(1, NUM_FOLDS + 1):

    for K in range(MIN_TOPICS, MAX_TOPICS + 1):

        model_dir = (
            MODEL_DIR /
            f"Fold_{fold}" /
            f"K_{K:03d}"
        )

        metadata_file = model_dir / "metadata.json"

        metadata = {}

        if metadata_file.exists():

            with open(
                metadata_file,
                "r",
                encoding="utf-8"
            ) as f:

                metadata = json.load(f)

            status = "Available"

        else:

            status = "Missing"

        manifest_rows.append({

            # =====================================================
            # Experiment Identification
            # =====================================================

            "ExperimentID": f"F{fold}_K{K:03d}",

            "Fold": fold,

            "Topics": K,

            "Status": status,

            # =====================================================
            # Dataset Information
            # =====================================================

            "TrainingDocuments":
                metadata.get("documents", np.nan),

            "DictionarySize":
                metadata.get("dictionary_size", np.nan),

            # =====================================================
            # LDA Hyperparameters
            # =====================================================

            "RandomSeed":
                metadata.get("random_seed", np.nan),

            "Passes":
                metadata.get("passes", np.nan),

            "Iterations":
                metadata.get("iterations", np.nan),

            "Alpha":
                metadata.get("alpha", np.nan),

            "Eta":
                metadata.get("eta", np.nan),

            # =====================================================
            # Training Statistics
            # =====================================================

            "TrainingTimeSeconds":
                metadata.get(
                    "training_time_seconds",
                    np.nan
                ),

            "LogPerplexityTrain":
                metadata.get(
                    "log_perplexity_train",
                    np.nan
                ),

            # =====================================================
            # Output Files
            # =====================================================

            "ModelDirectory":
                str(model_dir),

            "ModelFile":
                str(model_dir / "lda.model"),

            "MetadataFile":
                str(metadata_file),

            "ThetaTrainFile":
                str(model_dir / "theta_train.csv"),

            "ThetaTestFile": str(model_dir / "theta_test.csv"),


            "PhiFile":
                str(model_dir / "phi.csv"),

            "TopicSizesFile":
                str(model_dir / "topic_sizes.csv"),

            "TopWordsFile":
                str(model_dir / "top_words.csv")

        })

# ------------------------------------------------------------
# Create DataFrame
# ------------------------------------------------------------

manifest_df = pd.DataFrame(manifest_rows)

manifest_df = manifest_df.sort_values(
    by=["Fold", "Topics"]
).reset_index(drop=True)

# ------------------------------------------------------------
# Save Manifest
# ------------------------------------------------------------

manifest_file = (
    MODEL_DIR /
    "experiment_manifest.csv"
)

manifest_df.to_csv(
    manifest_file,
    index=False
)

# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print_line()

print("Master Experiment Manifest Created")

print()

print(f"Manifest File : {manifest_file}")

print(f"Experiments   : {len(manifest_df)}")

print(f"Available     : {(manifest_df['Status'] == 'Available').sum()}")

print(f"Missing       : {(manifest_df['Status'] == 'Missing').sum()}")

print_line()

display(manifest_df.head())

Master Experiment Manifest Created

Manifest File : /content/drive/MyDrive/TopicEvalBench/models/experiment_manifest.csv
Experiments   : 495
Available     : 495
Missing       : 0


,ExperimentID,Fold,Topics,Status,TrainingDocuments,DictionarySize,RandomSeed,Passes,Iterations,Alpha,...,TrainingTimeSeconds,LogPerplexityTrain,ModelDirectory,ModelFile,MetadataFile,ThetaTrainFile,ThetaTestFile,PhiFile,TopicSizesFile,TopWordsFile
0,F1_K002,1,2,Available,240,891,42,20,400,"[0.5, 0.5]",...,3.3123,-6.298535,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...
1,F1_K003,1,3,Available,240,891,42,20,400,"[0.3333333432674408, 0.3333333432674408, 0.333...",...,4.6282,-6.214306,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...
2,F1_K004,1,4,Available,240,891,42,20,400,"[0.25, 0.25, 0.25, 0.25]",...,2.7942,-6.193754,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...
3,F1_K005,1,5,Available,240,891,42,20,400,"[0.20000000298023224, 0.20000000298023224, 0.2...",...,6.0153,-6.191917,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...
4,F1_K006,1,6,Available,240,891,42,20,400,"[0.1666666716337204, 0.1666666716337204, 0.166...",...,2.5762,-6.224693,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...


In [151]:
print_line()

print("Notebook 3 Completed Successfully")

print()

print(f"Total Models : {len(overall_summary)}")

print()

print("Output Files Created")

print("""
✓ lda.model
✓ metadata.json
✓ theta_train.csv
✓ theta_test.csv
✓ phi.csv
✓ topic_sizes.csv
✓ top_words.csv
✓ training_summary.csv
✓ training_summary_all_folds.csv
✓ validation_report.csv
✓ experiment_manifest.csv
""")

print_line()

Notebook 3 Completed Successfully

Total Models : 495

Output Files Created

✓ lda.model
✓ metadata.json
✓ theta_train.csv
✓ theta_test.csv
✓ phi.csv
✓ topic_sizes.csv
✓ top_words.csv
✓ training_summary.csv
✓ training_summary_all_folds.csv
✓ validation_report.csv
✓ experiment_manifest.csv

